In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions
from textblob import TextBlob

C:\Users\vigne\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Load the document(.txt)

In [4]:
data=open('data.txt').read()

2. Text Normalization

In [5]:
data=data.lower()

In [6]:
data=re.sub(r'\s{2,}',' ',data)

In [7]:
import contractions
data=contractions.fix(data)

In [8]:
import emoji
data=emoji.replace_emoji(data,'')

In [9]:
import string
str_punc=string.punctuation.replace('.','')
data=data.translate(str.maketrans('','',str_punc))

In [10]:
data=re.sub(r'[^A-Za-z0-9\s\.]','',data)

In [11]:
values=TextBlob(data).correct()
values

TextBlob("1.machine learning my and deep a  learning do are both important subjects of artificial intelligence but they serve different purposes and approaches. let us break down each concept with realize examples  machine learning my definition
machine learning is a type of ai that allows computers to automatically learn and improve from experience without being explicitly programme. this means that algorithms can be trained to recognize patterns in data and make predictions or decisions. key concepts
 supervised learning where a model is trained on labelled data to predict outcome. example image classification distinguishing between different types of objects.
 supervised learning where the model finds patterns in data without being told what to look for. example fluttering grouping similar items together.
 reinforcement learning where an agent learns by interesting with an environment trying to minimize a reward. example selfdriving cars that learn to navigable and make decisions ba

Lemmatization

In [12]:
import spacy
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens).strip()
data

'1.machine learn ml deep   learning dl important subset artificial intelligence serve different purpose approach . let break concept realtime example   machine learning ml definition \n machine learning type ai allow computer automatically learn improve experience explicitly program . mean algorithm train recognize pattern datum prediction decision . key concept \n  supervise learning model train label datum predict outcome . example image classification distinguish different type object . \n  unsupervised learning model find pattern datum tell look . example cluster group similar item . \n  reinforcement learning agent learn interact environment try maximize reward . example selfdrive car learn navigate decision base feedback . realtime example \n imagine selfdriving car . drive safely learn experience . car use sensor like camera radar detect object obstacle realtime . use machine learn algorithm predict car base information collect . allow car informed decision slow stop change lane

Chunking

In [13]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
chunks=list(set(splitter.split_text(data)))

chunk embedding (covert chunks to vectors)

In [14]:
embedding_model = SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2' 
)
chunk_embeddings = embedding_model.encode(chunks).astype('float32')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2845.79it/s]


In [15]:
chunk_embeddings.shape

(15, 384)

In [16]:
dimension = chunk_embeddings.shape[1]
dimension

384

In [17]:
faiss.normalize_L2(chunk_embeddings)

In [18]:
index_faiss_db = faiss.IndexFlat(dimension)
index_faiss_db.add(chunk_embeddings)  # vector bd

retrival part

In [19]:
def r_search(query , k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1) #2d
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k) # retrival part
    R_chunk = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunk)
    return R_str

# result = r_search('what is big data')
# print (result)

# to generate the structured way we use llm 



In [20]:
print(r_search)

<function r_search at 0x000001F6096938A0>


gendration part

In [ ]:
def g_text(prompt):

    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model = 'gemini-3.5-flash',
        api_key = "apikey"
    )

    prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {prompt}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
    response = llm.invoke(prompt)
    return response.content[0]['text']
#send the request to gemini and get back the response
# response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
response = r_search(user_prompt)
g_response = g_text(response)
# print(g_response) 


with open('g_response.txt','w') as file:
    file.write(g_response)


(1, 384)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [22]:
# def g_text(r_search):
#         import os
#         import requests

#         API_URL = "https://router.huggingface.co/v1/chat/completions"
    
#         headers = {
#             "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
#         }
#         def query(payload):
#             response = requests.post(API_URL, headers=headers, json=payload)
#             return response.json()
#         prompt = f'''
#                     You're an helpful assistant
#                     Assigned Task for you : Structure my output => {r_search}
#                     Note : 
#                     1) Don't add extra contents just structure mentioned output.
#                     2) If there is mistake in output correct or else keep the original output
#                     with structured result.
#             '''
#         response = query({
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": f'{prompt}'
#                 }
#             ],
#             "model": "deepseek-ai/DeepSeek-R1:novita"
#         })
#         return response
# user_prompt = 'Explain Machine Learning ?'
# user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
# r_response = r_search(user_prompt)
# g_response = g_text(r_response)
# print(g_response) 